# Football Playing Style Analysis

This notebook combines team-level attacking, defensive, movement, passing, and physical statistics for the 2026 World Cup.
The goal is to identify distinct playing-style clusters using feature selection, scaling, PCA, and K-Means clustering.

We document each decision so the analysis is easy to follow and reproduce.


## 1. Setup and data loading

Import required libraries and load the five CSV files.
Rank columns are dropped because they are derived ordinal values rather than raw performance metrics.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


In [ ]:
attacking_data = pd.read_csv("worldcup_2026_team_attacking_stats.csv").drop("Rank", axis=1)
defensive_data = pd.read_csv("worldcup_2026_team_defensive_stats.csv").drop("Rank", axis=1)
movement_data = pd.read_csv("worldcup_2026_team_offers_receptions_stats.csv").drop("Rank", axis=1)
passing_data = pd.read_csv("worldcup_2026_team_passing_stats.csv").drop("Rank", axis=1)
physical_data = pd.read_csv("worldcup_2026_team_physical_stats.csv").drop("Rank", axis=1)


In [ ]:
merged_data = (
    attacking_data
    .merge(defensive_data, on="Team")
    .merge(movement_data, on="Team")
    .merge(passing_data, on="Team")
    .merge(physical_data, on="Team")
)


## 2. Inspect the merged dataset

Verify the merged data shape, columns, and missing values before selecting features for playing-style analysis.


In [ ]:
print("Merged data shape:", merged_data.shape)
print("Columns:", merged_data.columns.tolist())
display(merged_data.head())
print("Missing values by column:")
print(merged_data.isna().sum())


## 3. Select features representing playing style

Drop columns that describe outcomes, totals, or results rather than tactical style.
This keeps clustering focused on how teams play, not simply how many goals they scored or conceded.


In [ ]:
excluded_columns = [
    "Goals",
    "Assists",
    "Own goals",
    "Goals conceded",
    "xG efficiency",
    "Attempt at goal conversion rate",
    "Team",
    "Attempts on target",
    "Attempts off target",
    "Headed attempts at goal"
]
team_playing_style_data = merged_data.drop(excluded_columns, axis=1)
playing_style_features = team_playing_style_data.columns.tolist()
print("Selected playing-style features:", playing_style_features)


## 4. Scale the selected features

StandardScaler brings all features onto the same scale so that large-magnitude metrics do not dominate the clustering.


In [ ]:
scaler = StandardScaler()
scaled_playing_style_data = scaler.fit_transform(team_playing_style_data)


## 5. Reduce dimensionality with PCA

PCA reduces noise and makes it easier to visualize relationships between teams by projecting playing-style features into fewer components.


In [ ]:
pca = PCA(n_components=min(12, scaled_playing_style_data.shape[1]))
pca_data = pca.fit_transform(scaled_playing_style_data)
explained_variance = pca.explained_variance_ratio_
print("Explained variance ratio for the first components:", explained_variance.round(3))
print("Cumulative explained variance for the selected components:", explained_variance.cumsum().round(3))


SyntaxError: unterminated string literal (detected at line 4) (341545479.py, line 4)

## 6. Choose the number of clusters with the elbow method

The elbow method shows how inertia changes as the number of clusters increases.
We use this to select a cluster count that balances explanation and compactness.


In [ ]:
inertia = []
cluster_range = range(1, 11)
for k in cluster_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(pca_data)
    inertia.append(model.inertia_)
plt.figure(figsize=(8, 5))
plt.plot(cluster_range, inertia, marker="o")
plt.xlabel("Number of clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method for Cluster Selection")
plt.grid(True)
plt.show()


## 7. Fit K-Means and assign cluster labels

Using the selected number of clusters, we group teams with similar playing-style feature profiles.
Cluster labels are added back to the merged dataset for further exploration.


In [ ]:
n_clusters = 10
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(pca_data)
merged_data["Cluster"] = clusters
print("Cluster sizes:", merged_data["Cluster"].value_counts().sort_index())


## 8. Visualize team clusters in PCA space

Plot the first two principal components to inspect how teams group by playing style.
Annotating each point makes it easier to connect clusters with actual team identities.


In [ ]:
plt.figure(figsize=(14, 10))
scatter = plt.scatter(
    pca_data[:, 0],
    pca_data[:, 1],
    c=clusters,
    cmap="tab10",
    s=120,
    edgecolor="k"
)
for i, team in enumerate(merged_data["Team"]):
    plt.text(pca_data[i, 0], pca_data[i, 1], team, fontsize=8, alpha=0.8)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Football Team Playing Style Clusters")
plt.grid(True)
plt.colorbar(scatter, label="Cluster")
plt.show()


## 9. Analyze cluster profiles

Compute the average playing-style metrics for each cluster to understand the tactical characteristics captured by the model.


In [ ]:
cluster_profile = (
    merged_data.groupby("Cluster")[playing_style_features]
    .mean()
    .round(2)
)
display(cluster_profile)


## 10. Summary and rationale

- We merged all relevant team statistics to create a unified dataset for playing-style analysis.
- We excluded outcome-focused and aggregate columns so that clustering emphasizes tactical and stylistic metrics.
- Standard scaling was applied to prevent features with large magnitudes from dominating the model.
- PCA reduced dimensionality and produced a visual representation of playing-style similarity.
- K-Means clustering grouped teams into style cohorts, and cluster profiling reveals the typical tactical signatures for each group.

This structure makes the notebook easier to follow and supports reproducible interpretation of football playing styles.
